# 01 · Preprocessing

Inspect → harmonize → clean → optionally reserve an unseen attack → split → fit on train → transform/write each split in batches → release that split.

One dataset per run. Initial loading, splitting, and fitting still need RAM. Saved Parquet files contain ready-to-use transformed features.

## 0. Setup and Imports

In [21]:
PLATFORM = 'kaggle'  # 'colab' or 'kaggle'
assert PLATFORM in ('colab', 'kaggle')

In [22]:
if PLATFORM == 'colab':
    %load_ext cudf.pandas
    %load_ext cuml.accel

    from google.colab import drive
    drive.mount('/content/drive')
    INPATH = '/content/drive/MyDrive/Masters/Datasets/NF-UQ-NIDS(Splitted_Dataset)'
    OUTPATH = '/content/drive/MyDrive/Masters/IDS_repo/My repo'

else : 
    INPATH = '/kaggle/input/datasets/omarahmed001/ids-datasets'
    OUTPATH = '/kaggle/working/project'
    

In [23]:
%pip install -q "scikit-learn>=1.6"
from pathlib import Path
from fastai.tabular.all import df_shrink
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.base import clone
from sklearn.frozen import FrozenEstimator
from sklearn.preprocessing import MinMaxScaler, QuantileTransformer
from sklearn.decomposition import PCA


Note: you may need to restart the kernel to use updated packages.


In [24]:
DATA_DIR = Path(INPATH)
FILES = {
    'NF_UNSW_NB15': 'NF-UNSW-NB15-v2.parquet',
    'NF_BOT_IOT': 'NF-BoT-IoT-v2.parquet',
    'NF_CSE_CIC_IDS2018': 'NF-CSE-CIC-IDS2018-v2.parquet',
    'NF_TON_IOT': 'NF-ToN-IoT-v2.parquet',
    'CIC-BoT-IoT': 'CIC-BoT-IoT-V2.parquet',
    'CSE-CIC-IDS2018': 'CSE-CIC-IDS2018.parquet',
    'UNSW-NB15': 'UNSW-NB15.parquet'
}

In [25]:
DATASET = 'CIC-BoT-IoT'  # Change this to run one of the other datasets.
HOLD_OUT_ATTACK = False  # False: include every attack class in the 70/15/15 split.
HELD_OUT_ATTACK = 'Worms'  # Used only when HOLD_OUT_ATTACK is True.
SEED = 42

## 1. Load and inspect

In [26]:
df = pd.read_parquet(DATA_DIR / FILES[DATASET])
print(DATASET, df.shape)
display(df.head())
display(df['Attack'].value_counts())

CIC-BoT-IoT (11503556, 79)


,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,Attack
0,6,64465,0,2,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,1,Reconnaissance
1,6,22646986,3,1,300.0,100.0,100.0,100.0,100.0,0.0,...,3281987.0,0.0,3281987.0,3281987.0,9682500.0,5878777.50,13839423.0,5525576.0,1,DDoS
2,6,153463,1,2,0.0,100.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,1,Reconnaissance
3,17,24915470,4,1,0.0,0.0,0.0,0.0,0.0,0.0,...,4777031.0,0.0,4777031.0,4777031.0,7784938.5,2586274.75,9613711.0,5956166.0,1,DDoS
4,17,46437692,8,1,0.0,0.0,0.0,0.0,0.0,0.0,...,9597806.0,7235516.0,14714089.0,4481524.0,9080693.0,5048133.00,14796053.0,5230723.0,1,DDoS


Attack
DDoS              4786502
DoS               4781782
Reconnaissance    1896036
Benign              38435
Theft                 801
Name: count, dtype: int64

In [27]:
df.Label.value_counts()

Label
1    11465121
0       38435
Name: count, dtype: int64

In [28]:
with pd.option_context('display.max_rows', None):
    display(pd.DataFrame({
        'Dtype': df.dtypes,
        'Unique Values': df.nunique()
    }))

,Dtype,Unique Values
Protocol,int8,3
Flow Duration,int32,6106435
Total Fwd Packets,int32,1761
Total Backward Packets,int16,694
Fwd Packets Length Total,float32,6282
Bwd Packets Length Total,float32,2585
Fwd Packet Length Max,float32,876
Fwd Packet Length Min,float32,102
Fwd Packet Length Mean,float32,6752
Fwd Packet Length Std,float32,7530


- Multiple constant columns that will be dropped later

In [29]:
df['Down/Up Ratio'].value_counts()

Down/Up Ratio
0.0     9887718
1.0     1366432
2.0      242236
3.0        5192
4.0         785
7.0         460
6.0         261
5.0         223
8.0         187
9.0          52
11.0          6
10.0          3
13.0          1
Name: count, dtype: int64

### Actual sample versus the full original dataset

Keep up to `MAX_PER_CLASS = 300_000` randomly selected rows per `Attack`; retain every row from classes smaller than the cap. The cells compare this actual sample with the full original data before encoding, cleaning, splitting, or fitted transformations. No second reference sample or 50,000-row diagnostic cap is used.

The class table measures the deliberate change in class proportions. Numerical features are compared within each Attack using KS distance: 0 means matching empirical CDFs; 0.02 means a maximum CDF gap of 2 percentage points. Nominal fields use total variation distance between category frequencies (0 means matching frequencies). Missing and infinite rates are reported separately. Both distances range from 0 to 1; no universal pass/fail cutoff is imposed. Read the complete tables in `numeric_distribution_report` and `categorical_distribution_report`.

These are descriptive comparisons. KS p-values are not reported because the sample is a subset of the original, many features are discrete, and a small p-value would not establish a practically important difference. Marginal agreement does not prove matching correlations, rare subgroups, or tails. Keeping every minority row gives matching raw distributions for that class; it does not add new independent observations.

`CHECK_SAMPLE_DISTRIBUTION = False` skips the feature comparisons while retaining the class table and releasing the original. Full-data comparisons may take time; they process one column at a time and print each class. Rerun **Load and inspect** before repeating sampling.

Sampling remains before the split in this workflow: validation/test therefore reflect the capped class mix. For evaluation at the original class proportions, split the original data first and cap training only. Preprocessing is still fitted after splitting.

In [30]:
import gc

MAX_PER_CLASS = 300_000
CHECK_SAMPLE_DISTRIBUTION = True

# Retain the full original data until the comparison below is finished.
# This assignment does not copy the original DataFrame.
original_df = df
original_counts = original_df['Attack'].value_counts(dropna=False)

sampled_train = pd.concat(
    [
        group.sample(n=min(len(group), MAX_PER_CLASS), random_state=SEED)
        for _, group in original_df.groupby(
            'Attack', observed=True, dropna=False
        )
    ],
    ignore_index=True,
)
df = sampled_train.sample(frac=1, random_state=SEED).reset_index(drop=True)
del sampled_train

sample_counts = df['Attack'].value_counts(dropna=False)
class_comparison = pd.DataFrame({
    'Original rows': original_counts,
    'Sample rows': sample_counts,
}).fillna(0).astype('int64').rename_axis('Attack')
class_comparison['Original (%)'] = (
    100 * class_comparison['Original rows'] / len(original_df)
)
class_comparison['Sample (%)'] = (
    100 * class_comparison['Sample rows'] / len(df)
)
class_comparison['Difference (percentage points)'] = (
    class_comparison['Sample (%)'] - class_comparison['Original (%)']
)
class_comparison['Rows retained (%)'] = (
    100 * class_comparison['Sample rows'] / class_comparison['Original rows']
)
print(f'Original: {len(original_df):,} rows | Actual sample: {len(df):,} rows')
display(class_comparison.round(4))
class_tv_distance = (
    class_comparison['Sample (%)'] - class_comparison['Original (%)']
).abs().sum() / 200
print(f'Class-proportion TV distance: {class_tv_distance:.6f}')

Original: 11,503,556 rows | Actual sample: 939,236 rows


,Original rows,Sample rows,Original (%),Sample (%),Difference (percentage points),Rows retained (%)
Attack,,,,,,
Benign,38435,38435,0.3341,4.0922,3.7580,100.0000
DDoS,4786502,300000,41.6089,31.9409,-9.6680,6.2676
DoS,4781782,300000,41.5679,31.9409,-9.6270,6.2738
Reconnaissance,1896036,300000,16.4822,31.9409,15.4587,15.8225
Theft,801,801,0.0070,0.0853,0.0783,100.0000


Class-proportion TV distance: 0.192950


In [31]:
from scipy.stats import ks_2samp

# Nominal fields are compared by category frequencies, not numeric ordering.
# Add another dataset-specific nominal column here if needed.
sampling_nominal_columns = ['Protocol']

if CHECK_SAMPLE_DISTRIBUTION:
    categorical_features = [
        c for c in original_df.columns
        if c not in ('Label', 'Attack')
        and (c in sampling_nominal_columns
             or not pd.api.types.is_numeric_dtype(original_df[c]))
    ]
    numeric_features = [
        c for c in original_df.select_dtypes(include='number').columns
        if c not in ('Label', 'Attack') and c not in categorical_features
    ]

    numeric_results = []
    categorical_results = []

    for attack in original_counts.index:
        print(f'Comparing full original vs actual sample: {attack}', flush=True)
        original_mask = (original_df['Attack'].isna() if pd.isna(attack)
                         else original_df['Attack'].eq(attack))
        sample_mask = (df['Attack'].isna() if pd.isna(attack)
                       else df['Attack'].eq(attack))

        # Use every row in both datasets; process one feature at a time.
        for col in numeric_features:
            a = original_df.loc[original_mask, col].to_numpy(
                dtype=np.float64, na_value=np.nan
            )
            b = df.loc[sample_mask, col].to_numpy(
                dtype=np.float64, na_value=np.nan
            )
            finite_a = a[np.isfinite(a)]
            finite_b = b[np.isfinite(b)]

            distance = (
                ks_2samp(finite_a, finite_b, method='asymp').statistic
                if len(finite_a) and len(finite_b) else np.nan
            )
            numeric_results.append({
                'Attack': attack,
                'feature': col,
                'Original rows': len(a),
                'Sample rows': len(b),
                'KS_D': distance,
                'Original missing (%)': 100 * np.isnan(a).mean(),
                'Sample missing (%)': 100 * np.isnan(b).mean(),
                'Original infinite (%)': 100 * np.isinf(a).mean(),
                'Sample infinite (%)': 100 * np.isinf(b).mean(),
            })
            del a, b, finite_a, finite_b

        for col in categorical_features:
            original_frequencies = original_df.loc[
                original_mask, col
            ].value_counts(normalize=True, dropna=False)
            sample_frequencies = df.loc[
                sample_mask, col
            ].value_counts(normalize=True, dropna=False)
            frequencies = pd.concat([
                original_frequencies.rename('original'),
                sample_frequencies.rename('sample'),
            ], axis=1).fillna(0)
            tv_distance = (
                frequencies['original'] - frequencies['sample']
            ).abs().sum() / 2
            categorical_results.append({
                'Attack': attack,
                'feature': col,
                'TV_distance': tv_distance,
            })
            del original_frequencies, sample_frequencies, frequencies

        del original_mask, sample_mask

    numeric_distribution_report = pd.DataFrame(numeric_results, columns=[
        'Attack', 'feature', 'Original rows', 'Sample rows', 'KS_D',
        'Original missing (%)', 'Sample missing (%)',
        'Original infinite (%)', 'Sample infinite (%)',
    ]).sort_values('KS_D', ascending=False)
    categorical_distribution_report = pd.DataFrame(
        categorical_results, columns=['Attack', 'feature', 'TV_distance']
    ).sort_values('TV_distance', ascending=False)

    print('Largest numerical distribution differences (KS distance):')
    display(numeric_distribution_report.head(30))
    if not numeric_distribution_report.empty:
        display(numeric_distribution_report.groupby('Attack', dropna=False)[
            'KS_D'
        ].agg(['median', 'max']))
    print('Categorical distribution differences (total variation distance):')
    display(categorical_distribution_report)
else:
    print('Within-class distribution comparisons disabled.')

# Release the full original before encoding, splitting, and preprocessing.
# The sampled df and the small comparison reports remain available.
del original_df
gc.collect()
print('Sampling comparison finished; original data released from RAM.')

Comparing full original vs actual sample: DDoS
Comparing full original vs actual sample: DoS
Comparing full original vs actual sample: Reconnaissance
Comparing full original vs actual sample: Benign
Comparing full original vs actual sample: Theft
Largest numerical distribution differences (KS distance):


,Attack,feature,Original rows,Sample rows,KS_D,Original missing (%),Sample missing (%),Original infinite (%),Sample infinite (%)
98,DoS,Fwd IAT Max,4781782,300000,0.003004,0.0,0.0,0.0,0.0
21,DDoS,Fwd IAT Std,4786502,300000,0.003001,0.0,0.0,0.0,0.0
96,DoS,Fwd IAT Mean,4781782,300000,0.002550,0.0,0.0,0.0,0.0
151,DoS,Idle Min,4781782,300000,0.002535,0.0,0.0,0.0,0.0
111,DoS,Fwd Packets/s,4781782,300000,0.002393,0.0,0.0,0.0,0.0
16,DDoS,Flow IAT Std,4786502,300000,0.002380,0.0,0.0,0.0,0.0
168,Reconnaissance,Flow IAT Std,1896036,300000,0.002185,0.0,0.0,0.0,0.0
91,DoS,Flow IAT Mean,4781782,300000,0.002157,0.0,0.0,0.0,0.0
19,DDoS,Fwd IAT Total,4786502,300000,0.002135,0.0,0.0,0.0,0.0
93,DoS,Flow IAT Max,4781782,300000,0.002134,0.0,0.0,0.0,0.0


,median,max
Attack,,
Benign,0.000000,0.000000
DDoS,0.000545,0.003001
DoS,0.001406,0.003004
Reconnaissance,0.000912,0.002185
Theft,0.000000,0.000000


Categorical distribution differences (total variation distance):


,Attack,feature,TV_distance
1,DoS,Protocol,0.001491
0,DDoS,Protocol,0.000209
2,Reconnaissance,Protocol,0.000028
3,Benign,Protocol,0.000000
4,Theft,Protocol,0.000000


Sampling comparison finished; original data released from RAM.


In [32]:
# 1. Summarize the feature distances separately for every class.
# Multiply by 100 to express CDF gaps in percentage points.
numeric_summary = (
    numeric_distribution_report
    .groupby('Attack', observed=True, dropna=False)['KS_D']
    .agg(['median', 'max'])
    .mul(100)
    .rename(columns={
        'median': 'Median CDF gap across features (pp)',
        'max': 'Largest CDF gap (pp)',
    })
)

# Start with ALL classes, then attach their numerical results.
sample_distribution_summary = class_comparison[
    ['Original rows', 'Sample rows']
].join(numeric_summary)

print('SUMMARY — one row per class')
print('pp = percentage points; smaller gaps mean closer distributions.')

with pd.option_context(
    'display.max_rows', None,
    'display.max_columns', None,
):
    display(sample_distribution_summary.round(4))


# 2. Show five features for EACH class.
# This limits features per class, not the number of classes displayed.
for attack in sample_distribution_summary.index:
    mask = (
        numeric_distribution_report['Attack'].isna()
        if pd.isna(attack)
        else numeric_distribution_report['Attack'].eq(attack)
    )

    details = (
        numeric_distribution_report.loc[mask, ['feature', 'KS_D']]
        .sort_values('KS_D', ascending=False)
        .head(5)
        .copy()
    )
    details['Max CDF gap (pp)'] = 100 * details['KS_D']

    print(f'\n{attack} — five largest numerical feature differences:')
    display(details.reset_index(drop=True).round(6))


# 3. Show categorical results for ALL classes.
print('\nCATEGORY FREQUENCIES — zero means matching proportions.')

with pd.option_context('display.max_rows', None):
    display(
        categorical_distribution_report
        .sort_values(['Attack', 'feature'])
        .reset_index(drop=True)
        .round(6)
    )

SUMMARY — one row per class
pp = percentage points; smaller gaps mean closer distributions.


,Original rows,Sample rows,Median CDF gap across features (pp),Largest CDF gap (pp)
Attack,,,,
Benign,38435,38435,0.0000,0.0000
DDoS,4786502,300000,0.0545,0.3001
DoS,4781782,300000,0.1406,0.3004
Reconnaissance,1896036,300000,0.0912,0.2185
Theft,801,801,0.0000,0.0000



Benign — five largest numerical feature differences:


,feature,KS_D,Max CDF gap (pp)
0,Fwd Packet Length Min,0.0,0.0
1,Fwd Packet Length Mean,0.0,0.0
2,Fwd Packet Length Max,0.0,0.0
3,Bwd Packets Length Total,0.0,0.0
4,Total Backward Packets,0.0,0.0



DDoS — five largest numerical feature differences:


,feature,KS_D,Max CDF gap (pp)
0,Fwd IAT Std,0.003001,0.300118
1,Flow IAT Std,0.002380,0.237980
2,Fwd IAT Total,0.002135,0.213499
3,Fwd IAT Max,0.002060,0.206024
4,Idle Min,0.001914,0.191396


... (4 outputs left out)


Theft — five largest numerical feature differences:


,feature,KS_D,Max CDF gap (pp)
0,Fwd Avg Bytes/Bulk,0.0,0.0
1,Avg Bwd Segment Size,0.0,0.0
2,ECE Flag Count,0.0,0.0
3,CWE Flag Count,0.0,0.0
4,Down/Up Ratio,0.0,0.0



CATEGORY FREQUENCIES — zero means matching proportions.


,Attack,feature,TV_distance
0,Benign,Protocol,0.000000
1,DDoS,Protocol,0.000209
2,DoS,Protocol,0.001491
3,Reconnaissance,Protocol,0.000028
4,Theft,Protocol,0.000000


## 2. Encoding

| Feature | Unique values | Type | Handling | 
|---|---:|---|---:|
| `Protocol` | 3 | Nominal | One Hot Encoding

In [33]:
col_to_one_hot = [
    'Protocol'
]

IMPUTER_FIT_ROWS = 500_000
PCA_FIT_ROWS = IMPUTER_FIT_ROWS  # None: full train; integer: sample cap.
PCA_COMPONENTS = None  # None disables PCA; e.g. 10 reduces scaled measurements to 10 columns.
NUMERIC_SCALER = QuantileTransformer(output_distribution='normal', random_state=SEED)
PREPROCESSING_VERBOSE = True


In [34]:
col_categorical = col_to_one_hot 

## 3. Dataset Shrinking

In [35]:
import warnings
from fastai.tabular.core import df_shrink_dtypes

results = []

for col in df.select_dtypes(include="number").columns:
    # Inspect one column without copying or modifying the DataFrame.
    skip = [c for c in df.columns if c != col]

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", RuntimeWarning)

        proposed = df_shrink_dtypes(
            df,
            skip=skip,
            obj2cat=False,
            int2uint=False,
        )

    if caught:
        results.append({
            "column": col,
            "original_dtype": str(df[col].dtype),
            "proposed_dtype": str(proposed.get(col, df[col].dtype)),
            "minimum": df[col].min(),
            "maximum": df[col].max(),
            "warning": "; ".join(str(w.message) for w in caught),
        })

display(pd.DataFrame(results))

""


In [36]:
df.columns = df.columns.str.strip()
df['Attack'] = df['Attack'].astype("string[pyarrow]").str.strip()

In [37]:
df = df_shrink(df, obj2cat=False, int2uint=False)
with pd.option_context('display.max_rows', None):
    display(df.dtypes)

Protocol                               int8
Flow Duration                         int32
Total Fwd Packets                     int32
Total Backward Packets                int16
Fwd Packets Length Total            float32
Bwd Packets Length Total            float32
Fwd Packet Length Max               float32
Fwd Packet Length Min               float32
Fwd Packet Length Mean              float32
Fwd Packet Length Std               float32
Bwd Packet Length Max               float32
Bwd Packet Length Min               float32
Bwd Packet Length Mean              float32
Bwd Packet Length Std               float32
Flow Bytes/s                        float32
Flow Packets/s                      float32
Flow IAT Mean                       float32
Flow IAT Std                        float32
Flow IAT Max                        float32
Flow IAT Min                        float32
Fwd IAT Total                       float32
Fwd IAT Mean                        float32
Fwd IAT Std                     

## 4. Clean

In [38]:
# Identify constant columns
constant_cols = df.columns[df.nunique(dropna=False) == 1]

print("Constant columns:", constant_cols.tolist())
print("Number of constant columns:", len(constant_cols))

# Drop constant columns
#df.drop(columns=constant_cols, inplace=True)

Constant columns: ['Fwd PSH Flags', 'Fwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate', 'Init Fwd Win Bytes', 'Fwd Seg Size Min']
Number of constant columns: 10


In [39]:
# Drop constant and unnecessary columns
#df.drop(constant_cols)

In [40]:
print(df.duplicated().sum(), "fully duplicate rows to remove")

121 fully duplicate rows to remove


In [41]:
df.drop_duplicates(inplace=True)
df.reset_index(inplace=True, drop=True)

In [42]:
# Keep Label and Attack for splitting/evaluation; never feed either to the model.
df = df.loc[:, ~df.columns.str.lower().str.startswith(('unnamed:', 'index'))]
feature_cols = [c for c in df if c not in ('Label', 'Attack')]

# Protocol identifiers, DNS query types, and FTP return codes are nominal categories.
categorical_cols = [c for c in col_categorical if c in feature_cols]
for col in categorical_cols:
    df[col] = df[col].astype('string').fillna('missing').astype("string[pyarrow]").str.strip()
numeric_cols = [c for c in feature_cols if c not in categorical_cols]
df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
print(f'{len(numeric_cols)} numeric and {len(categorical_cols)} categorical features')

76 numeric and 1 categorical features


In [43]:
print('Categorical:', categorical_cols)
print('Numeric:', numeric_cols)

Categorical: ['Protocol']
Numeric: ['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Fwd Packets Length Total', 'Bwd Packets Length Total', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Min', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE F

In [44]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)    
print(df.isna().any(axis=1).sum(), "rows with at least one NaN to remove")
# df.dropna(inplace=True)

0 rows with at least one NaN to remove


## 5. Optionally reserve an unseen attack

In [45]:
# Remove incomplete targets and exact repeated rows before any split.
if HOLD_OUT_ATTACK:
    zero_day = df[df['Attack'].eq(HELD_OUT_ATTACK)]
    known = df[df['Attack'].ne(HELD_OUT_ATTACK)]
    print(f'Eligible: {len(known):,} | held-out {HELD_OUT_ATTACK}: {len(zero_day):,}')
else:
    known = df
    zero_day = df.iloc[:0]  # Keep the same schema; no zero-day file is saved.
    print(f'Holdout disabled: all {len(known):,} rows enter the normal split.')


Holdout disabled: all 939,115 rows enter the normal split.


## 6. Split eligible traffic (70% / 15% / 15%)

In [46]:
# Stratify eligible traffic by Attack; exclude the held-out attack only when enabled.
train, remainder = train_test_split(
    known, test_size=0.30, stratify=known['Attack'], random_state=SEED
)
val, test = train_test_split(
    remainder, test_size=0.50, stratify=remainder['Attack'], random_state=SEED
)
# Drop intermediate frames; retain only the four clean, untransformed splits.
del df, known, remainder
print('train / val / test / zero-day:', *(len(x) for x in (train, val, test, zero_day)))

train / val / test / zero-day: 657380 140867 140868 0


## Outliers

In [47]:
# Only numerical measurements—not categories or flag indicators.
outlier_cols = [
    c for c in numeric_cols
    if c not in col_to_one_hot
    and not c.startswith((
        'TCP_FLAGS_',
        'CLIENT_TCP_FLAGS_',
        'SERVER_TCP_FLAGS_',
    ))
]

rows = []

for col in outlier_cols:
    s = train[col]
    finite = s[np.isfinite(s)]

    q1, median, q3, p99, p999 = finite.quantile(
        [0.25, 0.50, 0.75, 0.99, 0.999]
    )
    iqr = q3 - q1

    # Zero IQR makes this rule uninformative for sparse/zero-heavy features.
    iqr_count = (
        ((finite < q1 - 1.5 * iqr) |
         (finite > q3 + 1.5 * iqr)).sum()
        if iqr > 0 else np.nan
    )

    rows.append({
        'feature': col,
        'missing': s.isna().sum(),
        'infinite': np.isinf(s).sum(),
        'minimum': finite.min(),
        'median': median,
        'p99': p99,
        'p99.9': p999,
        'maximum': finite.max(),
        'IQR': iqr,
        'IQR_flagged': iqr_count,
        'above_p99.9': (finite > p999).sum(),
        'outside_float32': (
            (finite > np.finfo(np.float32).max) |
            (finite < -np.finfo(np.float32).max)
        ).sum(),
        'outside_float64': (
            (finite > np.finfo(np.float64).max) |
            (finite < -np.finfo(np.float64).max)
        ).sum(),
    })

outlier_report = pd.DataFrame(rows).set_index('feature')
with pd.option_context('display.max_rows', None):
    display(outlier_report)

,missing,infinite,minimum,median,p99,p99.9,maximum,IQR,IQR_flagged,above_p99.9,outside_float32,outside_float64
feature,,,,,,,,,,,,
Flow Duration,0,0,1.000000,2.387003e+07,1.117658e+08,1.199856e+08,1.200000e+08,1.677090e+07,12518.0,658,0,0
Total Fwd Packets,0,0,0.000000,4.000000e+00,2.300000e+01,3.168315e+03,4.915200e+04,3.000000e+00,52889.0,658,0,0
Total Backward Packets,0,0,1.000000,1.000000e+00,5.000000e+00,2.521000e+03,3.251900e+04,1.000000e+00,37862.0,316,0,0
Fwd Packets Length Total,0,0,0.000000,1.000000e+02,8.060000e+02,2.507798e+06,1.864583e+08,3.000000e+02,6710.0,658,0,0
Bwd Packets Length Total,0,0,0.000000,1.000000e+02,4.940000e+02,2.391910e+06,1.960545e+08,1.000000e+02,15825.0,648,0,0
Fwd Packet Length Max,0,0,0.000000,1.000000e+02,5.170000e+02,1.458000e+03,6.424000e+04,1.000000e+02,13764.0,501,0,0
Fwd Packet Length Min,0,0,0.000000,0.000000e+00,1.000000e+02,5.320000e+02,1.448000e+03,1.000000e+02,3811.0,363,0,0
Fwd Packet Length Mean,0,0,0.000000,3.333333e+01,2.585000e+02,9.777026e+02,4.950624e+04,1.000000e+02,8403.0,657,0,0
Fwd Packet Length Std,0,0,0.000000,0.000000e+00,3.155569e+02,4.304191e+02,2.702002e+04,4.082483e+01,18724.0,658,0,0


## 8. Fit preprocessing on training data only

This CIC notebook previously had no PCA. `PCA_COMPONENTS = None` preserves its existing numeric outputs. To enable PCA over **all scaled numeric measurement columns**, choose an integer component count (no larger than the number of numeric features and fitting rows). Protocol one-hot columns stay outside PCA.

Set `PCA_FIT_ROWS = IMPUTER_FIT_ROWS` for sampled PCA, `None` for full-training PCA, or another positive cap. Pass these into `make_preprocessor(..., pca_fit_rows=PCA_FIT_ROWS, n_components=PCA_COMPONENTS)`. PCA changes the numeric output columns and requires retraining downstream models.

The CIC imputer and numeric scaler retain their existing sampled fitting behavior, now controlled by `IMPUTER_FIT_ROWS`. One-hot vocabularies include all observed training categories. Learned branches are frozen and assembled with one preview row, avoiding a combined full-training feature matrix. All fitting uses training rows only; transforms/writes remain batched. Full-train PCA mode can still require substantial RAM. Requires scikit-learn >= 1.6.


In [48]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

def category_fit_frame(data, columns):
    """Fit independent category vocabularies without expanding millions of rows."""
    values = {c: pd.Series(data[c]).drop_duplicates().to_numpy() for c in columns}
    longest = max(map(len, values.values()))
    # Repeat real values to align column lengths; do not invent missing categories.
    return pd.DataFrame({c: np.resize(v, longest) for c, v in values.items()})


def fit_pca(data, columns, preprocessing, n_components=1, fit_rows=None,
            random_state=42, verbose=True):
    """fit_rows=None uses all training rows; an integer caps PCA fitting rows."""
    if fit_rows is not None and fit_rows < 1:
        raise ValueError('fit_rows must be None or a positive row count.')
    if fit_rows is None or fit_rows >= len(data):
        fit_data = data[columns]
    else:
        fit_data = data.sample(n=fit_rows, random_state=random_state)[columns]
    if verbose:
        print(f'PCA fit: {len(fit_data):,}/{len(data):,} training rows; '
              f'{len(columns)} inputs -> {n_components} component(s).', flush=True)
    model = make_pipeline(
        *[step for _, step in preprocessing.steps],
        PCA(n_components=n_components, svd_solver='full'),
        verbose=verbose,
    )
    model.fit(fit_data)
    model.fit_rows_ = len(fit_data)
    if verbose:
        retained = model.steps[-1][1].explained_variance_ratio_.sum()
        print(f'PCA fitted; retained variance: {retained:.2%}.', flush=True)
    return model


def make_preprocessor(train, numeric_cols, one_hot, scaler=None,
                      imputer_fit_rows=500_000, pca_fit_rows=None,
                      n_components=None, random_state=42, verbose=True):
    """Optional PCA over scaled numeric measurements; categories remain separate."""
    sample = train.sample(
        n=min(imputer_fit_rows, len(train)), random_state=random_state,
    )[numeric_cols]
    imputer = IterativeImputer(max_iter=5, random_state=random_state, verbose=int(verbose))
    numeric_scaler = StandardScaler() if scaler is None else clone(scaler)
    numbers = make_pipeline(imputer, numeric_scaler, verbose=verbose)
    if verbose:
        print(f'Fitting CIC imputer/scaler on {len(sample):,} training rows...', flush=True)
    numbers.fit(sample)
    imputer_rows = len(sample)
    del sample

    # Preserve the original CIC numeric fit cap. PCA has its own independent cap.
    if n_components is not None:
        numbers_branch = fit_pca(
            train, numeric_cols, preprocessing=make_pipeline(FrozenEstimator(numbers)),
            n_components=n_components, fit_rows=pca_fit_rows,
            random_state=random_state, verbose=verbose,
        )
    else:
        numbers_branch = numbers
        if verbose:
            print('CIC PCA disabled (n_components=None).', flush=True)
    categories = OneHotEncoder(handle_unknown='ignore').fit(
        category_fit_frame(train, one_hot)
    )
    result = ColumnTransformer([
        ('numbers', FrozenEstimator(numbers_branch), numeric_cols),
        ('categories', FrozenEstimator(categories), one_hot),
    ], verbose=verbose)
    result.fit(train.iloc[:1])
    result.imputer_ = imputer
    result.numeric_scaler_ = numeric_scaler
    result.imputer_fit_rows_ = imputer_rows
    result.pca_ = numbers_branch.steps[-1][1] if n_components is not None else None
    if verbose:
        print(f'CIC preprocessor ready: {len(result.get_feature_names_out()):,} features.', flush=True)
    return result


preprocessor = make_preprocessor(
    train, numeric_cols, col_to_one_hot,
    scaler=NUMERIC_SCALER, imputer_fit_rows=IMPUTER_FIT_ROWS,
    pca_fit_rows=PCA_FIT_ROWS, n_components=PCA_COMPONENTS,
    random_state=SEED, verbose=PREPROCESSING_VERBOSE,
)


Fitting CIC imputer/scaler on 500,000 training rows...
[IterativeImputer] Completing matrix with shape (500000, 76)
[IterativeImputer] Change: 0.0, scaled tolerance: 917530.88 
[IterativeImputer] Early stopping criterion reached.
[Pipeline] .. (step 1 of 2) Processing iterativeimputer, total= 6.8min
[Pipeline]  (step 2 of 2) Processing quantiletransformer, total=   0.1s
CIC PCA disabled (n_components=None).
[IterativeImputer] Completing matrix with shape (1, 76)
[ColumnTransformer] ....... (1 of 2) Processing numbers, total=   0.0s
[ColumnTransformer] .... (2 of 2) Processing categories, total=   0.0s
CIC preprocessor ready: 79 features.


Save each split below in batches, then release its original frame. To repeat saving or run the optional raw-data benchmarks afterward, rerun the loading/splitting cells first.

## 9. Transform, save, and free each split

In [49]:
import pyarrow as pa

feature_names = preprocessor.get_feature_names_out().tolist()

float64_columns = {
    'numbers__SRC_TO_DST_SECOND_BYTES',
    'numbers__DST_TO_SRC_SECOND_BYTES',
}

original_target_schema = pa.Schema.from_pandas(
    train[['Label', 'Attack']], preserve_index=False
)
print(original_target_schema)
schema = pa.schema(
    [
        (
            name,
            pa.uint8() if name.startswith('categories__')
            else pa.float64() if name in float64_columns
            else pa.float32()
        )
        for name in feature_names
    ]
    + [
        original_target_schema.field('Label'),
        original_target_schema.field('Attack'),
    ]
)

print(f'Compact output schema:\n{schema}')

Label: int8
Attack: large_string
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 309
Compact output schema:
numbers__Flow Duration: float
numbers__Total Fwd Packets: float
numbers__Total Backward Packets: float
numbers__Fwd Packets Length Total: float
numbers__Bwd Packets Length Total: float
numbers__Fwd Packet Length Max: float
numbers__Fwd Packet Length Min: float
numbers__Fwd Packet Length Mean: float
numbers__Fwd Packet Length Std: float
numbers__Bwd Packet Length Max: float
numbers__Bwd Packet Length Min: float
numbers__Bwd Packet Length Mean: float
numbers__Bwd Packet Length Std: float
numbers__Flow Bytes/s: float
numbers__Flow Packets/s: float
numbers__Flow IAT Mean: float
numbers__Flow IAT Std: float
numbers__Flow IAT Max: float
numbers__Flow IAT Min: float
numbers__Fwd IAT Total: float
numbers__Fwd IAT Mean: float
numbers__Fwd IAT Std: float
numbers__Fwd IAT Max: float
numbers__Fwd IAT Min: float
numbers__Bwd IAT Total: float


import shutil
shutil.rmtree('/kaggle/working/project')

In [50]:
from datetime import datetime
import gc
import joblib
import sklearn
import pyarrow as pa
import pyarrow.parquet as pq
from scipy.sparse import issparse

PROJECT = Path(OUTPATH)
fitted_imputer = preprocessor.imputer_
fitted_numeric_scaler = preprocessor.numeric_scaler_
fitted_pca = preprocessor.pca_
attack_tag = f"holdout_{HELD_OUT_ATTACK}" if HOLD_OUT_ATTACK else "all_attacks"
RUN_NAME = f"{datetime.now():%Y%m%d}_{attack_tag}_{type(fitted_imputer).__name__}"
OUTPUT_DIR = PROJECT / 'outputs' / DATASET / 'preprocessing' / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BATCH_SIZE = 10000  # Reduce this if a transformed batch still uses too much RAM.
joblib.dump(preprocessor, OUTPUT_DIR / 'preprocessor.joblib', compress=0)

# Move frames into the queue; remove the other references so they can be freed.
splits = {'train': train, 'val': val, 'test': test}
if HOLD_OUT_ATTACK:
    splits['zero_day'] = zero_day
del train, val, test, zero_day
counts = []
for name in list(splits):
    rows = splits.pop(name)
    with pq.ParquetWriter(OUTPUT_DIR / f'{name}.parquet', schema, compression='zstd') as writer:
        for start in range(0, len(rows), BATCH_SIZE):
            batch = rows.iloc[start:start + BATCH_SIZE]
            X = preprocessor.transform(batch[feature_cols])
            # # Reduce batch memory before densifying sparse output.
            # # float32 rounding is intentional for compact model inputs.
            # X = X.astype(np.float32, copy=False)
            # Densify this small batch only; never the complete split.
            if issparse(X):
                X = X.toarray()
            prepared = pd.DataFrame(X, columns=feature_names)
            prepared['Label'] = batch['Label'].to_numpy()
            prepared['Attack'] = batch['Attack'].to_numpy()
            table = pa.Table.from_pandas(prepared, schema=schema, preserve_index=False)
            writer.write_table(table)
            del batch, X, prepared, table
    counts.append(f'- {name}: {len(rows):,} rows')
    print('Saved and released:', name, len(rows), 'rows')
    del rows
    gc.collect()  # Frees Python references; the allocator may retain some reserved RAM.

holdout_note = f"Reserve `{HELD_OUT_ATTACK}` as zero-day data" if HOLD_OUT_ATTACK else "Use all attack classes; no separate zero-day set"
saved_files = '`train.parquet`, `val.parquet`, `test.parquet`' + (', `zero_day.parquet`' if HOLD_OUT_ATTACK else '')
readme = f"""# {DATASET} preprocessing — {RUN_NAME}

Source: `{DATA_DIR / FILES[DATASET]}`
Holdout enabled: `{HOLD_OUT_ATTACK}`. Held-out attack: `{HELD_OUT_ATTACK if HOLD_OUT_ATTACK else "None (all attacks included)"}`. Seed: `{SEED}`. scikit-learn: `{sklearn.__version__}`.

## Sampling and comparison
Before cleaning and splitting, retain up to {MAX_PER_CLASS:,} random rows per Attack class and all rows from smaller classes. Compare the actual sample to the full original using class proportions, within-class numerical KS distance, and categorical total variation. Diagnostics do not use another reference sample. Validation/test reflect the capped class mix.

## Steps followed
1. Inspect data and attack counts; strip column/attack-name whitespace.
2. Remove dataset identifiers, IP addresses, and generated index columns.
3. Treat PROTOCOL, L7_PROTO, DNS_QUERY_TYPE, and FTP_COMMAND_RET_CODE as categories, fill missing categories with 'missing', and replace numeric infinities with NaN.
4. Remove rows missing targets and exact duplicate rows.
5. {holdout_note}; split eligible traffic 70/15/15, stratified by Attack. With holdout disabled, attack classes are eligible for every split; rare classes are not guaranteed to occur in each split.
6. Fit `{fitted_imputer!r}` and `{fitted_numeric_scaler!r}` on {preprocessor.imputer_fit_rows_:,} sampled training rows (cap {IMPUTER_FIT_ROWS:,}). One-hot vocabularies include all observed training categories. Optional numeric PCA: {PCA_COMPONENTS!r} components; requested PCA row cap: {PCA_FIT_ROWS!r}; actual PCA fit rows: {fitted_pca.n_samples_ if fitted_pca is not None else 'disabled'}. PCA variance retained: {fitted_pca.explained_variance_ratio_.sum() if fitted_pca is not None else 'disabled'}. All fitting uses training data only; branches are frozen and assembled with one preview row.
7. Transform each split in batches of at most {BATCH_SIZE:,} rows and write directly to Parquet with Zstandard compression.
8. Release each batch and completed split before continuing.

## Saved files
{saved_files} contain TRANSFORMED numeric feature columns plus `Label` and `Attack`.
Features are already imputed, encoded, and scaled: do not transform them again.
`preprocessor.joblib` is for new RAW data only. Feature order is `preprocessor.get_feature_names_out()`.
Storage types: scaled numerical features or optional numeric PCA outputs use float32; one-hot features use uint8. The configured SECOND_BYTES numeric outputs retain float64 when present. Protocol one-hot outputs remain outside PCA.
Label and Attack retain their original Arrow types. The same schema is used for every batch and split.
Float32 reduces precision compared with float64.
These files differ from older exports of cleaned, untransformed features.

{chr(10).join(counts)}

## Read prepared batches
```python
import pyarrow.parquet as pq
for record_batch in pq.ParquetFile('train.parquet').iter_batches(batch_size=10000):
    batch = record_batch.to_pandas()
    X = batch.drop(columns=['Label', 'Attack'])
    y = batch['Label']
    # Consume this batch directly; do not apply preprocessor.transform(X).
```

Initial loading/splitting and imputer fitting still require RAM. Full-train PCA can require substantial RAM; set PCA_FIT_ROWS to a smaller cap. To reload preprocessor.joblib, use scikit-learn >= 1.6; no custom transformer class is required. Saving releases the original split frames; rerun loading/splitting before using raw-data benchmarks or repeating the save step.
Row splits and exact deduplication do not guarantee session isolation or isolation of identical predictors with conflicting labels.
"""
(OUTPUT_DIR / 'README.md').write_text(readme, encoding='utf-8')
print('Prepared datasets saved to:', OUTPUT_DIR)

[IterativeImputer] Completing matrix with shape (10000, 76)
[IterativeImputer] Completing matrix with shape (10000, 76)
[IterativeImputer] Completing matrix with shape (10000, 76)
[IterativeImputer] Completing matrix with shape (10000, 76)
[IterativeImputer] Completing matrix with shape (10000, 76)
[IterativeImputer] Completing matrix with shape (10000, 76)
[IterativeImputer] Completing matrix with shape (10000, 76)
[IterativeImputer] Completing matrix with shape (10000, 76)
[IterativeImputer] Completing matrix with shape (10000, 76)
[IterativeImputer] Completing matrix with shape (10000, 76)
[IterativeImputer] Completing matrix with shape (10000, 76)
[IterativeImputer] Completing matrix with shape (10000, 76)
[IterativeImputer] Completing matrix with shape (10000, 76)
[IterativeImputer] Completing matrix with shape (10000, 76)
[IterativeImputer] Completing matrix with shape (10000, 76)
[IterativeImputer] Completing matrix with shape (10000, 76)
[IterativeImputer] Completing matrix wit

## 7. Class Imbalance

Load the processed `train.parquet`, resample its `Attack` classes with ADASYN, then clean the original majority Attack class with Tomek Links. Exclude both targets from features. Restore binary `Label` from the original Attack-to-Label mapping when saving. Validation and test are not resampled here. Synthetic one-hot coordinates can be fractional and remain floating point.

In [51]:
HANDEL_IMBALANCE = True
ADASYN_BALANCE_TOLERANCE = 0.01  # Leave classes within 1% unchanged.

In [52]:
if HANDEL_IMBALANCE:
    %pip install -q imbalanced-learn plotly

    import gc
    import numpy as np
    import pandas as pd
    import plotly.express as px
    from imblearn.over_sampling import ADASYN
    from imblearn.under_sampling import TomekLinks

    prepared_train = pd.read_parquet(OUTPUT_DIR / 'train.parquet')
    original_columns = prepared_train.columns.tolist()
    target_dtypes = prepared_train[['Label', 'Attack']].dtypes

    # Every Attack class must correspond to exactly one binary Label.
    target_pairs = prepared_train[['Attack', 'Label']].drop_duplicates()
    if target_pairs['Attack'].duplicated().any():
        raise ValueError('An Attack class has multiple Label values. Fix this mapping first.')
    attack_to_label = target_pairs.set_index('Attack')['Label']

    resampling_features = [
        c for c in prepared_train.columns if c not in ('Label', 'Attack')
    ]
    # Floating point preserves synthetic fractional one-hot values.
    X_train = prepared_train[resampling_features].to_numpy(dtype=np.float32)
    y_train = prepared_train['Attack'].to_numpy(copy=True)

    del prepared_train, target_pairs
    gc.collect()

    before_counts = pd.Series(y_train).value_counts().sort_index()
    majority_attack = before_counts.idxmax()

    print('Before resampling (Attack):')
    display(before_counts.rename_axis('Attack').rename('rows').to_frame())

Note: you may need to restart the kernel to use updated packages.
Before resampling (Attack):


,rows
Attack,
Benign,26848
DDoS,210000
DoS,210000
Reconnaissance,209971
Theft,561


ADASYN uses an explicit dictionary of desired class counts. Classes already within `ADASYN_BALANCE_TOLERANCE = 0.01` (1%) of the largest class are left unchanged. This avoids requesting a negligible top-up such as 29 samples across 209,971 Reconnaissance rows, where every per-row allocation can round to zero.

The 1% tolerance is an explicit policy, not a requirement of ADASYN. Smaller classes target the largest count; the generated counts remain approximate. There is no silent fallback to another sampler. Independent ADASYN failures, such as unsuitable neighborhoods, remain visible.

In [53]:
if HANDEL_IMBALANCE:
    target_count = int(before_counts.max())
    sampling_targets = {
        attack: target_count
        for attack, count in before_counts.items()
        if count < (1 - ADASYN_BALANCE_TOLERANCE) * target_count
    }

    if sampling_targets:
        smallest_target_class = min(
            int(before_counts[attack]) for attack in sampling_targets
        )
        if smallest_target_class < 2:
            raise ValueError(
                'ADASYN needs at least two examples in each targeted Attack class.'
            )

        print('ADASYN desired class counts:', sampling_targets, flush=True)
        adasyn = ADASYN(
            sampling_strategy=sampling_targets,
            n_neighbors=min(5, smallest_target_class - 1),
            random_state=SEED,
        )
        X_adasyn, y_adasyn = adasyn.fit_resample(X_train, y_train)
    else:
        print('Classes are already approximately balanced; skipping ADASYN.')
        X_adasyn, y_adasyn = X_train, y_train

    adasyn_counts = pd.Series(y_adasyn).value_counts().sort_index()
    print('After ADASYN (Attack):')
    display(pd.DataFrame({
        'Before': before_counts,
        'After ADASYN': adasyn_counts,
    }).fillna(0).astype(int).rename_axis('Attack'))

    del X_train, y_train
    gc.collect()

ADASYN desired class counts: {'Benign': 210000, 'Theft': 210000}
After ADASYN (Attack):


,Before,After ADASYN
Attack,,
Benign,26848,209520
DDoS,210000,210000
DoS,210000,210000
Reconnaissance,209971,209971
Theft,561,209973


In [54]:
if HANDEL_IMBALANCE:
    # Preserve the original policy: clean only the original majority class.
    tomek = TomekLinks(
        sampling_strategy=[majority_attack],
        n_jobs=-1,
    )
    X_train_balanced, y_train_balanced = tomek.fit_resample(
        X_adasyn, y_adasyn
    )

    print('Rows removed by Tomek Links:',
          len(y_adasyn) - len(y_train_balanced))

    after_counts = pd.Series(y_train_balanced).value_counts()
    distribution = pd.DataFrame({
        'Before': before_counts,
        'After ADASYN': adasyn_counts,
        'After Tomek Links': after_counts,
    }).fillna(0).astype(int).rename_axis('Attack')

    distribution['Before (%)'] = (
        100 * distribution['Before'] / distribution['Before'].sum()
    ).round(2)
    distribution['After (%)'] = (
        100 * distribution['After Tomek Links']
        / distribution['After Tomek Links'].sum()
    ).round(2)
    display(distribution)

    # Plot aggregated counts instead of passing millions of rows to Plotly.
    plot_counts = distribution[
        ['Before', 'After ADASYN', 'After Tomek Links']
    ].reset_index().melt(
        id_vars='Attack', var_name='Stage', value_name='Rows'
    )
    plot_counts['Attack'] = plot_counts['Attack'].astype(str)
    fig = px.histogram(
        plot_counts,
        x='Attack',
        y='Rows',
        color='Stage',
        histfunc='sum',
        barmode='group',
        title='Attack class distribution before and after resampling',
        labels={'Rows': 'Number of samples'},
        category_orders={'Attack': distribution.index.astype(str).tolist()},
    )
    fig.update_layout(yaxis_title='Number of samples', xaxis_tickangle=-45)
    fig.show()

    del X_adasyn, y_adasyn
    gc.collect()

Rows removed by Tomek Links: 67986


,Before,After ADASYN,After Tomek Links,Before (%),After (%)
Attack,,,,,
Benign,26848,209520,209520,4.08,21.35
DDoS,210000,210000,142014,31.94,14.47
DoS,210000,210000,210000,31.94,21.40
Reconnaissance,209971,209971,209971,31.94,21.39
Theft,561,209973,209973,0.09,21.39


In [55]:
if HANDEL_IMBALANCE:
    balanced_train = pd.DataFrame(
        X_train_balanced,
        columns=resampling_features,
    )
    balanced_train['Attack'] = pd.Series(y_train_balanced).astype(
        target_dtypes['Attack']
    )
    balanced_train['Label'] = balanced_train['Attack'].map(
        attack_to_label
    ).astype(target_dtypes['Label'])
    balanced_train = balanced_train[original_columns]

    balanced_path = OUTPUT_DIR / 'train_adasyn_tomek.parquet'
    balanced_train.to_parquet(
        balanced_path,
        index=False,
        compression='zstd',
    )
    print('Saved:', balanced_path)

    del balanced_train, X_train_balanced, y_train_balanced
    gc.collect()

Saved: /kaggle/working/project/outputs/CIC-BoT-IoT/preprocessing/20261110_all_attacks_IterativeImputer/train_adasyn_tomek.parquet


Train using `train_adasyn_tomek.parquet`, with the existing `val.parquet` and `test.parquet`. The saved training file contains both `Attack` and `Label`; use `Attack` as the multiclass target. Exclude both targets from the model inputs. These features are already preprocessed—do not transform them again.

The table and Plotly histograms above show counts before resampling, after ADASYN, and after Tomek Links. Percentages compare the original and final training distributions.

# Testing

Run the configuration cells at the top of the notebook (platform, paths, dataset and seed), then start here. You can also run this section after the main save cell has released its raw splits. Enable the desired tests below, then run the remaining Testing cells in order.

The first loading cell reloads the **same raw Parquet selected by `DATA_DIR / FILES[DATASET]`**, repeats the relevant feature preparation, and recreates the original 70/15/15 split. Only its training partition enters these experiments. It does not require the main fitted preprocessor or the main in-memory splits.

In [ ]:
# Enable either or both groups, then run the loading cell immediately below.
RUN_TESTING = True      